# Testing for models
Models :
- OLS Linear Regression 
- AR
- (ARMA)
- VAR
- (HAR)

Variables :
- Baseline : Stock price = f(oil price)
- Macro-state interaction : Stock price = f(oil price, D_macro_state) (CFNAI)
- Macro-state + shock type : Stock price = f(oil price, D_macro_state, D_shock_type)

CFNAI is monthly so our period reference will be monthly. 
Daily data will be aggregated to monthly by taking the last observation of the month.

In [20]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

## OLS regression

In [21]:

def ols(X,y):
    '''
    OLS regression : Stock price = f(oil price)
    '''
    X = sm.add_constant(X)
    model = sm.OLS(y, X).fit()

    return model.summary()

### Baseline

In [22]:
# Data preparation
daily_data = pd.read_excel("../data/raw/data_hec_project_1.xlsx", skiprows=5, sheet_name="Daily")
monthly_data = pd.read_excel("../data/raw/data_hec_project_1.xlsx", skiprows=5, sheet_name="Monthly")

daily_data.columns = daily_data.columns.astype(str).str.strip()
date_col = "Date" if "Date" in daily_data.columns else "Dates"
daily_data[date_col] = pd.to_datetime(daily_data[date_col], errors="coerce")
daily_data = daily_data.dropna(subset=[date_col]).set_index(date_col).sort_index()

SP = daily_data["SP500"].replace(["#N/A N/A", "NA", ""], pd.NA).apply(pd.to_numeric, errors="coerce")
WTI = daily_data["WTI"].replace(["#N/A N/A", "NA", ""], pd.NA).apply(pd.to_numeric, errors="coerce")

# Daily log-returns
log_returns_SP = np.log(SP).diff().dropna()
log_returns_WTI = np.log(WTI).diff().dropna()

print(ols(log_returns_WTI, log_returns_SP))

                            OLS Regression Results                            
Dep. Variable:                  SP500   R-squared:                       0.028
Model:                            OLS   Adj. R-squared:                  0.028
Method:                 Least Squares   F-statistic:                     268.2
Date:                Thu, 26 Mar 2026   Prob (F-statistic):           1.84e-59
Time:                        15:41:29   Log-Likelihood:                 29173.
No. Observations:                9442   AIC:                        -5.834e+04
Df Residuals:                    9440   BIC:                        -5.833e+04
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0003      0.000      2.664      0.0

### Macro-state interaction

In [23]:
# Monthly log-returns 
prices_m = pd.DataFrame({"SP500": SP, "WTI": WTI}).resample("ME").last().dropna()
log_returns_SP_monthly = np.log(prices_m["SP500"]).diff().dropna()
log_returns_WTI_monthly = np.log(prices_m["WTI"]).diff().dropna()

# CFNAI 
md = monthly_data.copy()
md.columns = md.columns.astype(str).str.strip()
md["Dates"] = pd.to_datetime(md["Dates"], errors="coerce")
md = md.dropna(subset=["Dates"])
md["Dates"] = md["Dates"].dt.to_period("M").dt.to_timestamp("M")
md = md.set_index("Dates").sort_index()

CFNAI = pd.to_numeric(md["CFNAI"].replace(["#N/A N/A", "NA", ""], pd.NA), errors="coerce")
D_exp = (CFNAI > 0).astype(int).rename("D_exp")
D_con = (CFNAI < -0.7).astype(int).rename("D_con")

# Regression
df_reg = pd.concat([
    log_returns_SP_monthly.rename("Y"),
    log_returns_WTI_monthly.rename("r_oil"),
    D_exp,
    D_con,
], axis=1).dropna()

df_reg["r_oil_x_D_exp"] = df_reg["r_oil"] * df_reg["D_exp"]
df_reg["r_oil_x_D_con"] = df_reg["r_oil"] * df_reg["D_con"]

X = df_reg[["r_oil", "r_oil_x_D_exp", "r_oil_x_D_con", "D_exp", "D_con"]]
y = df_reg["Y"]

print(ols(X, y))

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.053
Model:                            OLS   Adj. R-squared:                  0.042
Method:                 Least Squares   F-statistic:                     4.768
Date:                Thu, 26 Mar 2026   Prob (F-statistic):           0.000299
Time:                        15:41:29   Log-Likelihood:                 764.74
No. Observations:                 433   AIC:                            -1517.
Df Residuals:                     427   BIC:                            -1493.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0035      0.003      1.085

C:\Users\danny\AppData\Local\Temp\ipykernel_10216\3330033988.py:19: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  df_reg = pd.concat([


### Macro state + type of shocks

In [24]:
df3 = df_reg.copy()

# High yield and US treasury yields
HY   = daily_data["HY"].replace(["#N/A N/A", "NA", ""], pd.NA).apply(pd.to_numeric, errors="coerce")
US10 = daily_data["US10"].replace(["#N/A N/A", "NA", ""], pd.NA).apply(pd.to_numeric, errors="coerce")
US2  = daily_data["US2"].replace(["#N/A N/A", "NA", ""], pd.NA).apply(pd.to_numeric, errors="coerce")

HY_monthly   = HY.resample("ME").last()
US10_monthly = US10.resample("ME").last()
US2_monthly  = US2.resample("ME").last()

# CreditSpread and Slope
CreditSpread = (HY_monthly - US10_monthly).rename("CreditSpread")
Slope        = (US10_monthly - US2_monthly).rename("Slope")

df3 = df3.join(pd.concat([CreditSpread, Slope], axis=1), how="left")

# Load shock data
sh = pd.read_excel("../data/external/shock_types.xlsx")
sh.columns = sh.columns.astype(str).str.strip()
sh["Start"]      = pd.to_datetime(sh["Start"], errors="coerce")
sh["End"]        = pd.to_datetime(sh["End"], errors="coerce")
sh["Shock_type"] = pd.to_numeric(sh["Shock_type"], errors="coerce")
sh = sh.dropna(subset=["Start", "End", "Shock_type"]).sort_values(["Start", "End"])

# Monthly shock type series (0 = no shock, 1 = supply, 2 = demand, 3 = geo)
month_start = df3.index.to_period("M").to_timestamp("M")
shock_type = pd.Series(0, index=df3.index, name="Shock_type")
for _, row in sh.iterrows():
    mask = (month_start >= row["Start"]) & (month_start <= row["End"])
    shock_type.loc[mask] = int(row["Shock_type"])

# D_sup for supply, D_dem for demand, D_geo for geopolitical 
shock_dummies = pd.get_dummies(shock_type, prefix="D_shock", dtype=int)
shock_dummies = shock_dummies.drop(columns=["D_shock_0"] if "D_shock_0" in shock_dummies.columns else [])
shock_dummies = shock_dummies.rename(columns={"D_shock_1": "D_sup", "D_shock_2": "D_dem", "D_shock_3": "D_geo"})

df3 = df3.join(shock_dummies, how="left")
df3[shock_dummies.columns] = df3[shock_dummies.columns].fillna(0)

# r_oil * shock interactions
shock_cols = list(shock_dummies.columns)
for c in shock_cols:
    df3[f"r_oil_x_{c}"] = df3["r_oil"] * df3[c]

X3_cols = (
    ["r_oil", "r_oil_x_D_exp", "r_oil_x_D_con", "D_exp", "D_con"]
    + shock_cols
    + [f"r_oil_x_{c}" for c in shock_cols]
    + ["CreditSpread", "Slope"]
)
X3 = df3[X3_cols].dropna()
y3 = df3.loc[X3.index, "Y"]

print(ols(X3, y3))

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.133
Model:                            OLS   Adj. R-squared:                  0.107
Method:                 Least Squares   F-statistic:                     4.963
Date:                Thu, 26 Mar 2026   Prob (F-statistic):           3.79e-08
Time:                        15:41:29   Log-Likelihood:                 783.99
No. Observations:                 433   AIC:                            -1540.
Df Residuals:                     419   BIC:                            -1483.
Df Model:                          13                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0269      0.006      4.257

#### Results :
Significative : 
- r oil
- D contraction
- D shock geo
- Credit spread

Debatable :
- D shock demand

Not significative :
- D expansion
- D shock supply
- Interactions shocks exp and sup
- Slope

## AR Model

### Optimizing AR : solving for p 

In [25]:
p_max = 12
base = pd.concat([log_returns_SP_monthly.rename("Y"), log_returns_WTI_monthly.rename("r_oil")], axis=1).dropna()
rows = []
for p in range(1, p_max + 1):
    d = base.copy()
    for lag in range(1, p + 1):
        d[f"Y_lag{lag}"] = d["Y"].shift(lag)
    d = d.dropna()
    X_p = sm.add_constant(d[[f"Y_lag{lag}" for lag in range(1, p + 1)] + ["r_oil"]])
    m = sm.OLS(d["Y"], X_p).fit()
    rows.append({"p": p, "AIC": round(m.aic, 2), "BIC": round(m.bic, 2)})

res = pd.DataFrame(rows).set_index("p")
print(res.to_string())
print(f"\nBest p by AIC: {res['AIC'].idxmin()}")
print(f"Best p by BIC: {res['BIC'].idxmin()}")

        AIC      BIC
p                   
1  -1517.47 -1505.26
2  -1512.32 -1496.05
3  -1507.16 -1486.83
4  -1505.61 -1481.23
5  -1499.66 -1471.23
6  -1496.52 -1464.05
7  -1502.84 -1466.33
8  -1504.13 -1463.59
9  -1498.13 -1453.56
10 -1495.92 -1447.32
11 -1489.73 -1437.11
12 -1488.08 -1431.45

Best p by AIC: 1
Best p by BIC: 1


### Baseline

In [26]:
df_ar1 = pd.concat([
    log_returns_SP_monthly.rename("Y"),
    log_returns_WTI_monthly.rename("r_oil"),
], axis=1).dropna()
df_ar1["Y_lag1"] = df_ar1["Y"].shift(1)
df_ar1 = df_ar1.dropna()

X_ar1 = df_ar1[["Y_lag1", "r_oil"]]
y_ar1 = df_ar1["Y"]

print(ols(X_ar1, y_ar1))



                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.040
Model:                            OLS   Adj. R-squared:                  0.036
Method:                 Least Squares   F-statistic:                     9.003
Date:                Thu, 26 Mar 2026   Prob (F-statistic):           0.000148
Time:                        15:41:29   Log-Likelihood:                 761.74
No. Observations:                 433   AIC:                            -1517.
Df Residuals:                     430   BIC:                            -1505.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0068      0.002      3.348      0.0

### Macro state

In [27]:

df_ar2 = df_reg.copy()
df_ar2["Y_lag1"] = df_ar2["Y"].shift(1)
df_ar2 = df_ar2.dropna()

X_ar2 = df_ar2[["Y_lag1", "r_oil", "r_oil_x_D_exp", "r_oil_x_D_con", "D_exp", "D_con"]]
y_ar2 = df_ar2["Y"]

print(ols(X_ar2, y_ar2))

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.054
Model:                            OLS   Adj. R-squared:                  0.041
Method:                 Least Squares   F-statistic:                     4.052
Date:                Thu, 26 Mar 2026   Prob (F-statistic):           0.000580
Time:                        15:41:29   Log-Likelihood:                 762.76
No. Observations:                 432   AIC:                            -1512.
Df Residuals:                     425   BIC:                            -1483.
Df Model:                           6                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0037      0.003      1.152

### Macro state + shock types

In [28]:
df_ar3 = df3.copy()
df_ar3["Y_lag1"] = df_ar3["Y"].shift(1)

shock_cols_ar = [c for c in df_ar3.columns if c in ["D_sup", "D_dem", "D_geo"]]
shock_int_cols_ar = [f"r_oil_x_{c}" for c in shock_cols_ar if f"r_oil_x_{c}" in df_ar3.columns]

X_ar3_cols = ["Y_lag1", "r_oil", "r_oil_x_D_exp", "r_oil_x_D_con", "D_exp", "D_con"] + shock_cols_ar + shock_int_cols_ar + ["CreditSpread", "Slope"]
X_ar3 = df_ar3[X_ar3_cols].dropna()
y_ar3 = df_ar3.loc[X_ar3.index, "Y"]

print(ols(X_ar3, y_ar3))

model = sm.OLS(y_ar3, sm.add_constant(X_ar3)).fit()

from statsmodels.stats.diagnostic import acorr_ljungbox

acorr_ljungbox(model.resid, lags=[5], return_df=True)

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.147
Model:                            OLS   Adj. R-squared:                  0.118
Method:                 Least Squares   F-statistic:                     5.118
Date:                Thu, 26 Mar 2026   Prob (F-statistic):           6.54e-09
Time:                        15:41:29   Log-Likelihood:                 784.99
No. Observations:                 432   AIC:                            -1540.
Df Residuals:                     417   BIC:                            -1479.
Df Model:                          14                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0309      0.006      4.768

,lb_stat,lb_pvalue
5,11.504466,0.042246


#### Results :
Yt-1 significant, no other big changes than that, model is better in general, but ljung-box test says there is autocorrelation in residuals,
so model can be better.

## VAR Model
### Baseline

In [29]:
from statsmodels.tsa.api import VAR

var_data = pd.concat([
    log_returns_SP_monthly.rename("Y"),
    log_returns_WTI_monthly.rename("r_oil"),
], axis=1).dropna()

# Select lag order by AIC, max 12
var_model = VAR(var_data)
lag_sel = var_model.select_order(maxlags=12)
print(lag_sel.summary())

p_var = lag_sel.aic
var_result = var_model.fit(p_var)
print(var_result.summary())

 VAR Order Selection (* highlights the minimums)  
       AIC         BIC         FPE         HQIC   
--------------------------------------------------
0       -10.96     -10.94*   1.743e-05     -10.95*
1       -10.97      -10.92   1.717e-05      -10.95
2      -10.98*      -10.88  1.706e-05*      -10.94
3       -10.97      -10.83   1.728e-05      -10.91
4       -10.98      -10.81   1.708e-05      -10.91
5       -10.97      -10.76   1.726e-05      -10.88
6       -10.96      -10.71   1.742e-05      -10.86
7       -10.95      -10.66   1.755e-05      -10.84
8       -10.94      -10.62   1.771e-05      -10.81
9       -10.93      -10.56   1.800e-05      -10.78
10      -10.91      -10.51   1.822e-05      -10.75
11      -10.91      -10.47   1.830e-05      -10.73
12      -10.91      -10.44   1.820e-05      -10.73
--------------------------------------------------
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Thu, 26, Mar

#### Results
- Y is not autoregressive (Yt-1 is not significative)  
Roil : L1.y p = 0.572  
Roil : L2.y p = 0.354  
=> AR and HAR defavorable  

- Oil is autoregressive  
Roil : L1.r_oil p = 0.002  
Roil : L2.r_oil p = 0.034  
=> oil t = f(oil t-1, oil t-2, ...), could add past oil to explain present y changes.

- Y doesnt influence r_oil dynamically  
Y : L1.r_oil p = 0.063  
Y : L2.r_oil p = 0.142  
=> Past oil is NOT significative in explaining present y changes.  

Corr(Y,oil) = 0.225 -> there are factors that influence both simultaneously  
Can test model without dynamics, and model with just Oil t-1.  


### Testing for optimal regression

Only significative variables

In [30]:
cols = ["Y", "r_oil", "D_con", "D_geo", "CreditSpread"]

df_model = df3[cols].copy()
df_model["r_oil_x_D_geo"] = df_model["r_oil"] * df_model["D_geo"]


df_model = df_model.dropna()

X = df_model[["r_oil", "D_con", "D_geo", "r_oil_x_D_geo", "CreditSpread"]]
y = df_model["Y"]


X_const = sm.add_constant(X)
model = sm.OLS(y, X_const).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.117
Model:                            OLS   Adj. R-squared:                  0.106
Method:                 Least Squares   F-statistic:                     11.29
Date:                Thu, 26 Mar 2026   Prob (F-statistic):           3.06e-10
Time:                        15:41:29   Log-Likelihood:                 779.86
No. Observations:                 433   AIC:                            -1548.
Df Residuals:                     427   BIC:                            -1523.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0278      0.005      5.580

Treat all dummies (Macro state and shocks) as only neutral or not

In [ ]:
cols = ["Y", "r_oil", "D_exp", "D_con", "D_sup", "D_dem", "D_geo", "CreditSpread"]

df_model2 = df3[cols].copy()

df_model2["D_macro"] = ((df_model2["D_exp"] == 1) | (df_model2["D_con"] == 1)).astype(int)

df_model2["D_shock"] = (
    (df_model2["D_sup"] == 1)
    | (df_model2["D_dem"] == 1)
    | (df_model2["D_geo"] == 1)
).astype(int)

df_model2["r_oil_x_D_shock"] = df_model2["r_oil"] * df_model2["D_shock"]

use_cols = ["Y", "r_oil", "D_macro", "D_shock", "r_oil_x_D_shock", "CreditSpread"]
df_model2 = df_model2[use_cols].dropna()

X2 = df_model2[["r_oil", "D_macro", "D_shock", "r_oil_x_D_shock", "CreditSpread"]]
y2 = df_model2["Y"]

model2 = sm.OLS(y2, sm.add_constant(X2)).fit()
print(model2.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.093
Model:                            OLS   Adj. R-squared:                  0.083
Method:                 Least Squares   F-statistic:                     8.792
Date:                Thu, 26 Mar 2026   Prob (F-statistic):           5.93e-08
Time:                        15:48:52   Log-Likelihood:                 774.19
No. Observations:                 433   AIC:                            -1536.
Df Residuals:                     427   BIC:                            -1512.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                      coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------------
const               0.0226      0.005     